# Objectives of this lab

In this lab, we will:
- Walk through how a Chainlink oracle works
- Participate in our own oracle contract that collects values from the class
- Demonstrate resilience against adversarial values

Have **more than 0.02 Sepolia ETH** for two reports plus transaction gas. Follow the TA's cue before processing either round.


In [ ]:
from web3 import Web3

# You may replace this with your own Sepolia RPC URL
web3 = Web3(Web3.HTTPProvider("https://ethereum-sepolia-rpc.publicnode.com"))
assert web3.eth.chain_id == 11155111, "Use Ethereum Sepolia"


# Exploring production oracles (Chainlink):

[Chainlink](https://chain.link/) is one of the most popular oracle providers on Ethereum. Let's take a high-level look at how the smart contracts for [the ETH-USD oracle on Chainlink](https://data.chain.link/ethereum/mainnet/crypto-usd/eth-usd) work for an example of how oracles can be implemented.

An end-to-end view of a price feed such as this consists of three types of contracts:
1. A consumer contract
2. A proxy contract
3. An aggregator contract

### Consumer contract

The consumer contract is any piece of code that queries the aggregator for the oracle values. For the assignments, we would be writing a contract in Solidity. Below is a sample code for doing the same in Python.

In [ ]:
# ETH/USD Chainlink data feed address (Sepolia)
ETH_USD_PRICE_FEED_ADDR = '0x694AA1769357215DE4FAC081bf1f309aDC325306'

# AggregatorV3Interface ABI
abi = '[{"inputs":[],"name":"decimals","outputs":[{"internalType":"uint8","name":"","type":"uint8"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"description","outputs":[{"internalType":"string","name":"","type":"string"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint80","name":"_roundId","type":"uint80"}],"name":"getRoundData","outputs":[{"internalType":"uint80","name":"roundId","type":"uint80"},{"internalType":"int256","name":"answer","type":"int256"},{"internalType":"uint256","name":"startedAt","type":"uint256"},{"internalType":"uint256","name":"updatedAt","type":"uint256"},{"internalType":"uint80","name":"answeredInRound","type":"uint80"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"latestRoundData","outputs":[{"internalType":"uint80","name":"roundId","type":"uint80"},{"internalType":"int256","name":"answer","type":"int256"},{"internalType":"uint256","name":"startedAt","type":"uint256"},{"internalType":"uint256","name":"updatedAt","type":"uint256"},{"internalType":"uint80","name":"answeredInRound","type":"uint80"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"version","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"}]'

# Set up contract instance
contract = web3.eth.contract(address=ETH_USD_PRICE_FEED_ADDR, abi=abi)

# Make call to latestRoundData()
latest_round_data = contract.functions.latestRoundData().call() # returned format is roundId, answer, startedAt, updatedAt, answeredInRound

price = latest_round_data[1] / 10**contract.functions.decimals().call()
print(f"Current oracle reported ETH-USD price: {price}")

### Proxy contract

Proxy contracts are on-chain proxies that point to the aggregator for a particular data feed. Using proxies enables the underlying aggregator to be upgraded without any service interruption to consuming contracts.

The proxy contract for ETH-USD is deployed [here on mainnet](https://etherscan.io/address/0x5f4ec3df9cbd43714fe2740f5e3616155c5b8419#code) and [here on sepolia](https://sepolia.etherscan.io/address/0x694AA1769357215DE4FAC081bf1f309aDC325306).

### Aggregator contract

The contract code for the aggregator is given [here](https://github.com/smartcontractkit/libocr/blob/master/contract/AccessControlledOffchainAggregator.sol). An aggregator is the contract that receives periodic data updates from the network of nodes that have agreed to post their data to a particular Chainlink feed. The data collection and aggregation is done off-chain and the data is then posted on-chain. Aggregators store aggregated data on-chain so that consumers can retrieve it and and act upon it within the same transaction.

You can access this data using the Data Feed address and the [AggregatorV3Interface contract](https://github.com/smartcontractkit/chainlink/blob/b5883718b03b99d46c67ee3a1fe2b3abf3e74385/contracts/src/v0.6/interfaces/AggregatorV3Interface.sol).

Aggregators receive updates from the oracle network only when the *Deviation Threshold* or *Heartbeat Threshold* triggers an update during an aggregation round. The first condition that is met triggers an update to the data.

- Deviation Threshold: A new aggregation round starts when a node identifies that the off-chain values deviate by more than the defined deviation threshold from the on-chain value. Individual nodes monitor one or more data providers for each feed.
- Heartbeat Threshold: A new aggregation round starts after a specified amount of time from the last update.

# Experimenting with our own oracle contract

The contract at `0xba87209026dAb94F9AA9f3326810205AA2df82DF` is an example of a very simple decentralized oracle implementation we created. Note that it is designed purely for an educational example and should not be used in production. There are many gas inefficiencies and obvious potential attack vectors, including a trivial Sybil attack.

The oracle works as follows:
- The main functions are: `readValue()`, `reportValue(uint256)`, `processUpdate()`
- Anyone can call `readValue()` to get the current oracle value and the block number that the value was last updated
- Anyone can call `reportValue(uint256)` to report a value for the next round must pay `REQUIRED_PAYMENT` to do so
- Anyone can call `processUpdate()` after `UPDATE_INTERVAL` blocks has passed since the last update
    - The caller receives 10% of all payments in the round as an incentive
    - The median value of all reported values is chosen as the latest oracle value
    - All accounts that reported the chosen value split the remaining 90% of funds
    
This is a simple example design of an oracle that incentivises people to report the "correct" value, as they will be rewarded for doing so.

The deployed version also exposes `reportCount()` and `reports(index)` for monitoring. For an even number of reports, it uses the upper median. If a wallet rejects an ETH payout, its reward is stored in `pendingRewards(account)` and can be claimed with `claimReward()`. Matching the median does not prove the reported value is true.


Below is the source for the class oracle. Compile `ClassOracle` with Solidity 0.8.26 in Remix, then use **At Address** with `0xba87209026dAb94F9AA9f3326810205AA2df82DF` on Sepolia. Everyone must use this shared oracle; do not deploy a separate instance. You can also use the [contract on Etherscan](https://sepolia.etherscan.io/address/0xba87209026dAb94F9AA9f3326810205AA2df82DF#code) once its source is verified.


```solidity
// SPDX-License-Identifier: MIT
pragma solidity 0.8.26;
import "@openzeppelin/contracts@4.9.6/security/ReentrancyGuard.sol";

/// Classroom median oracle. The 15-block interval starts at the last update.
contract ClassOracle is ReentrancyGuard {
    struct ValueReport { address reporter; uint256 value; }
    uint256 public constant REQUIRED_PAYMENT = 0.01 ether;
    uint256 public constant UPDATE_INTERVAL = 15;
    uint256 public constant MAX_REPORTS = 200;
    ValueReport[] public reports;
    uint256 public latestValue;
    uint256 public lastUpdateBlock;
    uint256 public round = 1;
    mapping(address => uint256) private reportedRound;
    mapping(address => uint256) public pendingRewards;
    event Reported(uint256 indexed round, address indexed reporter, uint256 value);
    event Updated(uint256 indexed round, uint256 median, uint256 count, address indexed caller, uint256 callerReward);
    constructor() { lastUpdateBlock = block.number; }
    function readValue() external view returns (uint256, uint256) { return (latestValue, lastUpdateBlock); }
    function reportCount() external view returns (uint256) { return reports.length; }
    function hasReported(address account) external view returns (bool) { return reportedRound[account] == round; }
    function reportValue(uint256 value) external payable nonReentrant {
        require(msg.value == REQUIRED_PAYMENT, "Send exactly 0.01 ETH");
        require(reportedRound[msg.sender] != round, "Already reported this round");
        require(reports.length < MAX_REPORTS, "Round is full");
        reportedRound[msg.sender] = round;
        reports.push(ValueReport(msg.sender, value));
        emit Reported(round, msg.sender, value);
    }
    function processUpdate() external nonReentrant {
        require(block.number >= lastUpdateBlock + UPDATE_INTERVAL, "Wait 15 blocks from last update");
        uint256 n = reports.length;
        require(n > 0, "No reports");
        ValueReport[] memory current = reports;
        uint256[] memory values = new uint256[](n);
        for (uint256 i; i < n; ++i) {
            uint256 v = current[i].value; uint256 j = i;
            while (j > 0 && values[j - 1] > v) { values[j] = values[j - 1]; --j; }
            values[j] = v;
        }
        uint256 median = values[n / 2]; // Upper median when n is even.
        uint256 correct;
        for (uint256 i; i < n; ++i) if (current[i].value == median) ++correct;
        uint256 pot = n * REQUIRED_PAYMENT;
        uint256 callerReward = pot / 10;
        uint256 reporterReward = (pot - callerReward) / correct;
        uint256 finishedRound = round;
        latestValue = median; lastUpdateBlock = block.number;
        ++round; delete reports;
        _pay(msg.sender, callerReward);
        for (uint256 i; i < n; ++i) if (current[i].value == median) _pay(current[i].reporter, reporterReward);
        emit Updated(finishedRound, median, n, msg.sender, callerReward);
    }
    function _pay(address recipient, uint256 amount) private {
        (bool ok,) = payable(recipient).call{value: amount, gas: 30000}("");
        if (!ok) pendingRewards[recipient] += amount;
    }
    function claimReward() external nonReentrant {
        uint256 amount = pendingRewards[msg.sender]; require(amount > 0, "No pending reward");
        pendingRewards[msg.sender] = 0;
        (bool ok,) = payable(msg.sender).call{value: amount}(""); require(ok, "Reward transfer failed");
    }
}
```


In [ ]:
ORACLE_CONTRACT_ADDRESS = "0xba87209026dAb94F9AA9f3326810205AA2df82DF"
ABI = '[{"inputs":[],"name":"REQUIRED_PAYMENT","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"UPDATE_INTERVAL","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"processUpdate","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"readValue","outputs":[{"internalType":"uint256","name":"","type":"uint256"},{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint256","name":"value","type":"uint256"}],"name":"reportValue","outputs":[],"stateMutability":"payable","type":"function"}]'

# Read the latest oracle value

Let's read the latest value from the oracle.

In [ ]:
contract = web3.eth.contract(address=ORACLE_CONTRACT_ADDRESS, abi=ABI)
value, block_number = contract.functions.readValue().call()

print(f"The current oracle value is {value}")
print(f"Value was last updated at block {block_number} ({web3.eth.block_number - block_number} blocks ago)")

## Reporting a value to the oracle

Now, let's report a value to the oracle for the upcoming update. Agree on a question and a common integer scale with the TA first.

Remember, everyone who reports the median value during this round will split all the rewards, so it is in your interest to report a value as close as possible to what you think is the correct value. You can only make one report per round.

- Load the shared `ClassOracle` in Remix using **At Address** as described above. If verified, its [Etherscan write contract page](https://sepolia.etherscan.io/address/0xba87209026dAb94F9AA9f3326810205AA2df82DF#writeContract) also works.
- Call `reportValue(value)` with your integer report and **exactly 0.01 Sepolia ETH** (set Remix Value to `0.01` and its unit to `ether`). Reset Remix Value to **0** before other calls.

Whenever at least `UPDATE_INTERVAL` blocks have passed since the last update, anyone can call `processUpdate()` to trigger the oracle update and rewards distribution. This example contract distributes Sepolia ETH as a reward, but you could easily imagine that the reward could be the protocol's token or something else similar instead.

Wait for the TA to confirm everyone has reported before calling `processUpdate()`. Its caller reward is before gas. The 15-block minimum runs from the **previous update**, not from the first report, so an eligible round can close immediately after a report. Calling `readValue()` tells you the last update block.

## Reporting adversarial values

Oracles should be resilient to malicious input. For this part, we want approximately 1/3 of the class to report an adversarial value and hopefully demonstrate that our oracle still produces a reasonable value.

When instructed to do so:
- If your birth month is January, February, March or April
    - Call `reportValue()` and report an adversarial value
- Otherwise
    - Call `reportValue()` and report what you believe is the true value

The TA should ensure fewer than half of the actual reports are adversarial; adjust these groups if necessary. Wait for everyone to report before processing.


After the next update, call `readValue()` to see what value the oracle decided on. Hopefully, it seems reasonable!